# Chapter 5 - Neural Network from Scratch (live)

Companion notebook for [Chapter 5](../part-2-deep-learning/05-neural-networks-from-scratch.md). The book derives backprop on a scalar autograd engine; here we build the **vectorized** version in NumPy and *watch it learn*: the loss curve falling and a non-linear **decision boundary** forming on the two-moons dataset.

**What you'll see:** a 2-16-16-1 MLP trained with manual backprop + Adam separates two interleaving moons that no straight line can split.

> Requires only `numpy` and `matplotlib` (`pip install numpy matplotlib`). Pure from-scratch - no autograd framework.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
np.random.seed(0)

## 1. A dataset no line can split: two moons

In [ ]:
def make_moons(n=400, noise=0.20, seed=0):
    rng = np.random.default_rng(seed)
    n1 = n // 2; n2 = n - n1
    t1 = np.pi * rng.random(n1)
    moon0 = np.c_[np.cos(t1), np.sin(t1)]
    t2 = np.pi * rng.random(n2)
    moon1 = np.c_[1 - np.cos(t2), 1 - np.sin(t2) - 0.5]
    X = np.r_[moon0, moon1] + noise * rng.standard_normal((n, 2))
    y = np.r_[np.zeros(n1), np.ones(n2)]
    return X.astype(np.float64), y.astype(np.float64)

X, y = make_moons()
X = (X - X.mean(0)) / X.std(0)            # standardize -> easier optimization

plt.figure(figsize=(5, 4))
plt.scatter(X[:, 0], X[:, 1], c=y, cmap='coolwarm', s=18, edgecolors='k', linewidths=0.3)
plt.title('Two moons - not linearly separable'); plt.xlabel('x1'); plt.ylabel('x2')
plt.tight_layout(); plt.show()

## 2. A 2-layer MLP with manual backprop

`x -> Linear -> tanh -> Linear -> tanh -> Linear -> sigmoid`, trained with binary cross-entropy. The backward pass is the chain rule by hand - exactly what `.backward()` automates.

In [ ]:
class Adam:
    """Minimal Adam over a dict of numpy arrays (updated in place)."""
    def __init__(self, params, lr=1e-2, b1=0.9, b2=0.999, eps=1e-8):
        self.p, self.lr, self.b1, self.b2, self.eps = params, lr, b1, b2, eps
        self.m = {k: np.zeros_like(v) for k, v in params.items()}
        self.v = {k: np.zeros_like(v) for k, v in params.items()}
        self.t = 0
    def step(self, grads):
        self.t += 1
        for k in self.p:
            self.m[k] = self.b1 * self.m[k] + (1 - self.b1) * grads[k]
            self.v[k] = self.b2 * self.v[k] + (1 - self.b2) * grads[k] ** 2
            mh = self.m[k] / (1 - self.b1 ** self.t)
            vh = self.v[k] / (1 - self.b2 ** self.t)
            self.p[k] -= self.lr * mh / (np.sqrt(vh) + self.eps)

In [ ]:
def init_params(sizes=(2, 16, 16, 1), seed=0):
    rng = np.random.default_rng(seed)
    p = {}
    for i in range(len(sizes) - 1):
        fan_in = sizes[i]
        p[f'W{i}'] = rng.standard_normal((sizes[i], sizes[i + 1])) * np.sqrt(1.0 / fan_in)
        p[f'b{i}'] = np.zeros(sizes[i + 1])
    return p

def forward(p, X):
    z0 = X @ p['W0'] + p['b0']; a0 = np.tanh(z0)
    z1 = a0 @ p['W1'] + p['b1']; a1 = np.tanh(z1)
    z2 = a1 @ p['W2'] + p['b2']
    prob = 1 / (1 + np.exp(-z2))                       # sigmoid output
    cache = (X, z0, a0, z1, a1, prob)
    return prob.ravel(), cache

def backward(p, cache, y):
    X, z0, a0, z1, a1, prob = cache
    N = len(y)
    dz2 = (prob - y[:, None]) / N                       # BCE + sigmoid -> p - y
    g = {}
    g['W2'] = a1.T @ dz2; g['b2'] = dz2.sum(0)
    da1 = dz2 @ p['W2'].T; dz1 = da1 * (1 - a1 ** 2)
    g['W1'] = a0.T @ dz1; g['b1'] = dz1.sum(0)
    da0 = dz1 @ p['W1'].T; dz0 = da0 * (1 - a0 ** 2)
    g['W0'] = X.T @ dz0; g['b0'] = dz0.sum(0)
    return g

def bce(prob, y, eps=1e-9):
    return -np.mean(y * np.log(prob + eps) + (1 - y) * np.log(1 - prob + eps))

## 3. Train and watch the loss fall

In [ ]:
params = init_params()
opt = Adam(params, lr=0.02)
losses = []
for step in range(3000):
    prob, cache = forward(params, X)
    losses.append(bce(prob, y))
    opt.step(backward(params, cache, y))

print('start loss %.3f -> final loss %.4f' % (losses[0], losses[-1]))
prob, _ = forward(params, X)
print('train accuracy: %.3f' % ((prob > 0.5).astype(float) == y).mean())

plt.figure(figsize=(5, 3.5))
plt.plot(losses); plt.title('Training loss (BCE)'); plt.xlabel('step'); plt.ylabel('loss')
plt.tight_layout(); plt.show()

## 4. The decision boundary it learned

In [ ]:
xx, yy = np.meshgrid(np.linspace(X[:, 0].min() - 0.5, X[:, 0].max() + 0.5, 300),
                     np.linspace(X[:, 1].min() - 0.5, X[:, 1].max() + 0.5, 300))
grid = np.c_[xx.ravel(), yy.ravel()]
zz, _ = forward(params, grid)
zz = zz.reshape(xx.shape)

plt.figure(figsize=(5.5, 4.5))
plt.contourf(xx, yy, zz, levels=20, cmap='coolwarm', alpha=0.7)
plt.contour(xx, yy, zz, levels=[0.5], colors='k', linewidths=1.5)
plt.scatter(X[:, 0], X[:, 1], c=y, cmap='coolwarm', s=18, edgecolors='k', linewidths=0.3)
plt.title('Learned non-linear decision boundary'); plt.xlabel('x1'); plt.ylabel('x2')
plt.tight_layout(); plt.show()

## Takeaway

Hidden layers + a non-linearity (`tanh`) let the net **bend** the decision surface around the moons - something no linear model can do. The whole engine is just forward matmuls and a hand-written chain-rule backward; `.backward()` in PyTorch does exactly this, automatically, at scale.

**Try it:** remove the non-linearities (use identity instead of `tanh`) and watch the boundary collapse to a straight line. Or shrink the hidden width to 2 and see it underfit.

[Back to Chapter 5](../part-2-deep-learning/05-neural-networks-from-scratch.md) | [Solutions](../solutions/05-neural-networks-solutions.md)